In [0]:
df= spark.table("bronze.customers")
display(df)

CustomerId,CustomerName,Email,City,Age
1,Saravana,saravana@gmail.com,Chennai,30
2,John,john@gmail.com,Bangalore,28
3,Alice,alice@gmail.com,Hyderabad,25
4,Bob,bob@gmail.com,Chennai,35
5,David,david@gmail.com,Coimbatore,40
6,Emma,emma@gmail.com,Madurai,27
7,Raj,raj@gmail.com,Salem,31
7,Raj,raj@gmail.com,Salem,31
8,Kumar,kumar@gmail.com,chennai,29
9,Anita,null,Chennai,29


In [0]:
df.printSchema()

root
 |-- CustomerId: integer (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Age: integer (nullable = true)



In [0]:
df.count()

11

In [0]:
from pyspark.sql.functions import col, trim, initcap
df = df.withColumn("CustomerName", trim(initcap(col("CustomerName"))))
df = df.withColumn("City", trim(initcap(col("City"))))
df = df.withColumn("Email", trim(col("Email")))

In [0]:
df=df.withColumn("Age", col("Age").cast("int"))

In [0]:
display(df.filter(col("Age").isNull()))

CustomerId,CustomerName,Email,City,Age


In [0]:
df=df.dropDuplicates(["CustomerId"])

In [0]:
df.groupBy("CustomerId") \
    .count() \
    .filter(col("count") > 1) \
    .show()

+----------+-----+
|CustomerId|count|
+----------+-----+
+----------+-----+



In [0]:
df=df.filter(
    col("CustomerId").isNotNull()&
    col("CustomerName").isNotNull()&
    col("Email").isNotNull()& 
    col("City").isNotNull()&
    col("Age").isNotNull()
)

In [0]:
df.printSchema()
df.count()

root
 |-- CustomerId: integer (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Age: integer (nullable = true)



8

In [0]:
%sql
CREATE SCHEMA IF not EXISTS silver;

In [0]:
df.write \
    .mode("overwrite") \
    .saveAsTable("silver.customers")

In [0]:
%sql
select * from silver.customers where age >30;
select City,Count(CustomerId) from silver.customers group by City;

City,Count(CustomerId)
Coimbatore,1
Chennai,3
Hyderabad,1
Bangalore,1
Madurai,1
Salem,1


In [0]:
df_filter=df.select("CustomerId","CustomerName","City")
df_filter.show()

+----------+------------+----------+
|CustomerId|CustomerName|      City|
+----------+------------+----------+
|         1|    Saravana|   Chennai|
|         2|        John| Bangalore|
|         3|       Alice| Hyderabad|
|         4|         Bob|   Chennai|
|         5|       David|Coimbatore|
|         6|        Emma|   Madurai|
|         7|         Raj|     Salem|
|         8|       Kumar|   Chennai|
+----------+------------+----------+

